# 01 · Dwell statistics and outlier months

Before detecting anything, check that each month of raw dwells looks like the others.
Months with sudden drops in panel size or activity (vendor changes, outages) are
flagged and removed from the analysis — in the paper this produced the list in
`aggregation.months` of `configs/regions/gushdan.json`.

Weekends and holidays follow the Israeli calendar (`calendar` settings); time periods,
quantiles and outlier thresholds are in the `dwell_stats` block.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from remote_work_utils import dwell_stats
from remote_work_utils.settings import load_settings, resolve_path

settings = load_settings("../configs/demo.json")
months = settings["aggregation"]["months"]
raw = {month: pd.read_parquet(resolve_path(settings, "dwells_raw", month=month)) for month in months}

In [ ]:
monthly = pd.DataFrame([dwell_stats.monthly_dwell_stats(raw[m], m, settings) for m in months])
monthly.set_index("month").T.round(3)

## Outlier months
A statistic is an outlier for a month when it falls more than `n_std` standard deviations
below the median of all months (or is a degenerate 0 / 1). With only three synthetic
months nothing is removed; degenerate shares such as "all users have 20+ dwells" are
listed so that they can be dropped from the comparison.

In [ ]:
dwell_stats.outlier_months(monthly, settings["dwell_stats"]["outliers"])

## Activity by time of day
`share_dwells_non_workday` rises in April 2020 (Passover and the first lockdown both fall in
this month in the synthetic calendar).

In [ ]:
periods = [w["name"] for w in settings["dwell_stats"]["time_periods"]] + [settings["dwell_stats"]["other_time_period"]]
shares = monthly.set_index("month")[[f"share_dwells_{p}" for p in periods]] * 100
shares.columns = periods

ax = shares.plot(kind="bar", stacked=True, figsize=(7, 4), colormap="viridis", rot=0)
ax.set_title("Dwells by start time of day and month")
ax.set_xlabel("Month (YYYYMM)")
ax.set_ylabel("Share of dwells (%)")
ax.legend(title="Start time", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

> **Fig. 1.1.** Share of dwells by local start time, per month. The profile is stable across
> months, so no month is excluded. *Notes:* synthetic data; periods from `dwell_stats.time_periods`.